In [1]:
import os
import json
import pandas as pd

In [2]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
KEY= os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [5]:
model = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    provider= "featherless-ai",
    max_new_tokens=1500,
    stop=["\n[1]", "\n\n["],
    repetition_penalty=1.03,
    huggingfacehub_api_token=KEY
)

WARNING! stop is not default parameter.
                    stop was transferred to model_kwargs.
                    Please make sure that stop is what you intended.
c:\Users\PARNA JAIN\mcqgen\mcqenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
llm = ChatHuggingFace(llm=model, temperature=0.3)

In [7]:
from langchain_core.prompts import PromptTemplate
import PyPDF2

In [8]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [9]:
TEMPLATE = """
Text: {text}
You are an expert MCQ maker. Given the above text, create EXACTLY {number} multiple choice questions for {subject} students in {tone} tone.
Respond with ONLY the JSON object below, filled in — no explanations, no citations, no text before or after the JSON.
Every single question, MUST include a "correct" key indicating the right answer (a, b, c, or d). A question without a "correct" key is invalid and unacceptable.
Make sure the questions are not repeated and check all the questions for grammar and spelling mistakes and to be conforming the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \

### RESPONSE_JSON
{response_json}
"""

In [10]:
quiz_generator_template = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template= TEMPLATE
)

In [11]:
quiz_chain= quiz_generator_template|llm | StrOutputParser()

In [12]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [13]:
quiz_evaluation_prompt = PromptTemplate(
    input_variables=["subject", "quiz"],
    template=TEMPLATE2
)

In [14]:
review_chain = quiz_evaluation_prompt | llm | StrOutputParser()

In [15]:
generate_evaluate_chain = (
    RunnablePassthrough.assign(quiz=quiz_chain)
    | RunnablePassthrough.assign(review=review_chain)
)

In [16]:
file_path= r"C:\Users\PARNA JAIN\mcqgen\data.txt"

In [17]:
with open(file_path, "r") as file:
    text = file.read()

In [18]:
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [19]:
number= 5
subject= "Artificial Intelligence"
tone= "simple"

In [21]:
response= generate_evaluate_chain.invoke(
    {
        "text": text,
        "number": number,
        "subject": subject,
        "tone": tone,
        "response_json": json.dumps(RESPONSE_JSON),
    }
)

In [22]:
response

{'text': "Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]\n\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g. images, audio, and videos).\n\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state s

In [23]:
quiz= response.get("quiz")

In [24]:
print(len(quiz))
print(quiz)

2125
{"1": {"mcq": "Which of the following is a primary goal of AI research?", "options": {"a": "Learning", "b": "Reasoning", "c": "Knowledge representation", "d": "Planning"}, "correct": "All of the above are primary goals of AI research."},

"2": {"mcq": "Which of the following is a high-profile application of AI?", "options": {"a": "Advanced web search engines", "b": "Chatbots", "c": "Virtual assistants", "d": "Autonomous vehicles"}, "correct": "All of the above are high-profile applications of AI."},

"3": {"mcq": "Which of the following is a traditional goal of AI research?", "options": {"a": "Learning", "b": "Reasoning", "c": "Knowledge representation", "d": "Planning"}, "correct": "All of the above are traditional goals of AI research."},

"4": {"mcq": "Which of the following is a technique used to develop and study methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving de

In [25]:
import re

def extract_mcqs(text):
    result = {}
    for match in re.finditer(r'"(\d+)":\s*\{', text):
        key = match.group(1)
        start = match.end() - 1
        depth = 0
        for i in range(start, len(text)):
            if text[i] == '{':
                depth += 1
            elif text[i] == '}':
                depth -= 1
                if depth == 0:
                    value_str = text[start:i+1]
                    try:
                        result[key] = json.loads(value_str)
                    except json.JSONDecodeError:
                        pass
                    break
    return result

quiz_json = extract_mcqs(quiz)
print(len(quiz_json))

5


In [26]:
quiz_json

{'1': {'mcq': 'Which of the following is a primary goal of AI research?',
  'options': {'a': 'Learning',
   'b': 'Reasoning',
   'c': 'Knowledge representation',
   'd': 'Planning'},
  'correct': 'All of the above are primary goals of AI research.'},
 '2': {'mcq': 'Which of the following is a high-profile application of AI?',
  'options': {'a': 'Advanced web search engines',
   'b': 'Chatbots',
   'c': 'Virtual assistants',
   'd': 'Autonomous vehicles'},
  'correct': 'All of the above are high-profile applications of AI.'},
 '3': {'mcq': 'Which of the following is a traditional goal of AI research?',
  'options': {'a': 'Learning',
   'b': 'Reasoning',
   'c': 'Knowledge representation',
   'd': 'Planning'},
  'correct': 'All of the above are traditional goals of AI research.'},
 '4': {'mcq': 'Which of the following is a technique used to develop and study methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maxi

In [27]:
quiz_table_data = []
for key, value in quiz_json.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
            ]
        )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

In [28]:
quiz_table_data

[{'MCQ': 'Which of the following is a primary goal of AI research?',
  'Choices': 'a: Learning | b: Reasoning | c: Knowledge representation | d: Planning',
  'Correct': 'All of the above are primary goals of AI research.'},
 {'MCQ': 'Which of the following is a high-profile application of AI?',
  'Choices': 'a: Advanced web search engines | b: Chatbots | c: Virtual assistants | d: Autonomous vehicles',
  'Correct': 'All of the above are high-profile applications of AI.'},
 {'MCQ': 'Which of the following is a traditional goal of AI research?',
  'Choices': 'a: Learning | b: Reasoning | c: Knowledge representation | d: Planning',
  'Correct': 'All of the above are traditional goals of AI research.'},
 {'MCQ': 'Which of the following is a technique used to develop and study methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals?',
  'Choices': 'a: State space search |

In [29]:
quiz_df = pd.DataFrame(quiz_table_data)

In [30]:
quiz_df

,MCQ,Choices,Correct
0,Which of the following is a primary goal of AI...,a: Learning | b: Reasoning | c: Knowledge repr...,All of the above are primary goals of AI resea...
1,Which of the following is a high-profile appli...,a: Advanced web search engines | b: Chatbots |...,All of the above are high-profile applications...
2,Which of the following is a traditional goal o...,a: Learning | b: Reasoning | c: Knowledge repr...,All of the above are traditional goals of AI r...
3,Which of the following is a technique used to ...,a: State space search | b: Mathematical optimi...,All of the above are techniques used to develo...
4,Which of the following is a major concern for ...,a: AI safety and unintended consequences and h...,All of the above are major concerns for resear...


In [31]:
quiz_df.to_csv("ai_quiz.csv", index=False)